In [ ]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
def get_initial_data(random_state=42):
    '''
    missing_corrected.csv를 불러오고,
    데이터 스플릿까지 하는 함수
    Args:
        random_state(int, optional): 데이터 스플릿할 때 필요한 랜덤 시드
    '''
    CURDIR = os.getcwd()
    DATA_PATH = os.path.join(CURDIR, 'missing_corrected.csv')
    DATA = pd.read_csv(DATA_PATH)
    DATA.head()

    # 범주형 변수 더미화 시 train/test 간 불일치를 방지하기 위해
    # 스플릿 전에 전체 데이터 기준으로 가능한 범주를 정의하고 CategoricalDtype으로 고정
    for col in DATA.columns:
        cats = list(DATA[col].unique())
        DATA[col] = DATA[col].astype(pd.api.types.CategoricalDtype(categories=cats))

    # validation set, test set이 imputation 설계하는 데 들어가면 안 됨, 일종의 사후판단 정보가 들어갈 수 있기 때문
    y = DATA['REASONb']
    X = DATA.drop('REASONb', axis=1)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
data = get_initial_data()
X_train = data[0]
X_train.shape


In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd
import numpy as np

def best_n_neighbors(X, y, neighbor_candidates=[3, 5, 7, 10, 15, 20]):
    """
    여러 n_neighbors 값을 테스트하여 평균 Mutual Information이 가장 높은 값을 선택한다.
    """
    result = {}

    for k in neighbor_candidates:
        mi = mutual_info_classif(
            X,
            y,
            discrete_features=True,
            n_neighbors=k,
            random_state=42
        )
        result[k] = np.mean(mi)   # 각 k에 대한 MI 평균 저장

    # 가장 높은 MI 평균을 가지는 k 선택
    best_k = max(result, key=result.get)

    return best_k, result


# 사용 예시
'''
from sklearn.feature_selection import mutual_info_classif
nostfips = X_train.drop('STFIPS', axis=1)
stfips = X_train['STFIPS']

mi = mutual_info_classif(nostfips, stfips, discrete_features=True, n_neighbors=3, random_state=42)
mi_series = pd.Series(mi, index=nostfips.columns).sort_values(ascending=False)

best_k, result_dict = best_n_neighbors(nostfips, stfips)
print("Best n_neighbors:", best_k)
print("MI mean by k:", result_dict)
'''


In [ ]:
from sklearn.feature_selection import mutual_info_classif
mi_list = {}

for col in X_train.columns:
    print(col)
    x = X_train.drop(col, axis=1)
    y = X_train[col]

    best_k, result_dict = best_n_neighbors(x, y)
    print("Best n_neighbors:", best_k)

    mi = mutual_info_classif(x, y, discrete_features=True, n_neighbors=best_k, random_state=42)

    mi_list[col] = mi
    print(f'{col} finished')


In [ ]:
mi_dict = {}
for col in X_train.columns:
    x_col = X_train.drop(col, axis=1)
    x_col = x_col.columns
    mi_series = pd.Series(mi_list[col], index=x_col)
    mi_dict[col] = mi_series

In [ ]:
import pickle
with open("mi_dict.pickle", 'wb') as f:
    pickle.dump(mi_dict, f)

{'STFIPS': DIVISION    2.101302
 CBSA2020    2.091105
 REGION      1.381283
 PRIMPAY     0.868864
 HLTHINS     0.759915
               ...   
 METHFLG     0.001397
 HALLFLG     0.001151
 BARBFLG     0.000482
 INHFLG      0.000272
 TRNQFLG     0.000169
 Length: 72, dtype: float64,
 'EDUC': STFIPS     0.362169
 LIVARAG    0.306288
 SUB1       0.289132
 ROUTE1     0.286771
 EMPLOY     0.278598
              ...   
 AMPHFLG    0.000222
 METHFLG    0.000208
 OTCFLG     0.000196
 INHFLG     0.000115
 TRNQFLG    0.000035
 Length: 72, dtype: float64}

In [ ]:
'''
짧게 핵심만 줄게.

## 시작값 추천

* 변수 개수 `p` 기준 휴리스틱: **k = ceil(log2 p)** 를 시작점으로 잡고, **[3, 20]** 범위로 클리핑.

* p < 30 → k = 2–4
* 50 ≤ p ≤ 200 → k = 5–10
* p ≥ 500 → k = 10–20

## 왜 이렇게?

* 너무 작으면 그래프가 끊김(성능↓), 너무 크면 **완전그래프처럼 비용↑/노이즈↑**.
* `log p` 수준은 **연결성**과 **희소성**의 균형을 주는 보편적 출발점.

## 실제 적용 팁

1. **대칭(top-k mutual)**: 서로가 상대를 top-k로 뽑았을 때만 연결 → 허브 과밀 완화.
2. **연결성 체크**: 컴포넌트가 많으면 k를 소폭↑(예: +2).
3. **그리드 스윕**: {3,5,8,10,15,20} 중에서 **검증 성능**(F1/AUROC/Loss)으로 선택.
4. **가중치 유지**: edge_attr에 MI를 두면 k를 조금 낮춰도 정보 보전.
5. **드롭 급락점**: 각 변수의 MI 시리즈에서 **기울기 급락 지점**을 k로 잡는 데이터-적응 방식도 효과적.

요약: **k = ceil(log2 p)**로 시작해서, **연결성/검증성능/계산비용**을 보며 {±2} 범위에서 빠르게 튜닝하는 걸 권장.
'''

In [22]:
import pickle
with open('mi_dict.pickle', 'rb') as f:
    mi_dict = pickle.load(f)
mi_dict

{'STFIPS': EDUC        0.362169
 MARSTAT     0.345997
 SERVICES    0.298998
 DETCRIM     0.228849
 LOS         0.225272
               ...   
 DIVISION    2.101302
 REGION      1.381283
 IDU         0.326536
 ALCDRUG     0.342647
 CBSA2020    2.091105
 Length: 72, dtype: float64,
 'EDUC': STFIPS      0.362169
 MARSTAT     0.077297
 SERVICES    0.045566
 DETCRIM     0.025137
 LOS         0.023396
               ...   
 DIVISION    0.214416
 REGION      0.131015
 IDU         0.259281
 ALCDRUG     0.271851
 CBSA2020    0.188498
 Length: 72, dtype: float64,
 'MARSTAT': STFIPS      0.345997
 EDUC        0.077297
 SERVICES    0.040065
 DETCRIM     0.025217
 LOS         0.008253
               ...   
 DIVISION    0.174774
 REGION      0.144743
 IDU         0.066566
 ALCDRUG     0.071681
 CBSA2020    0.234939
 Length: 72, dtype: float64,
 'SERVICES': STFIPS      0.298998
 EDUC        0.045566
 MARSTAT     0.040065
 DETCRIM     0.038683
 LOS         0.276616
               ...   
 DIVISION    0

In [23]:
import torch

def build_edge_index_from_mi_directed(mi_dict, top_k=6, return_edge_attr=False):
    """
    mi_dict: {col_name: pd.Series} (index=다른 변수명, value=MI, 내림차순 정렬)
    top_k: 각 src에서 MI 상위 k개로만 유향 엣지(src->dst) 생성
    return_edge_attr: True면 edge_attr로 MI 가중치 반환
    """
    cols = list(mi_dict.keys())
    col_to_idx = {c: i for i, c in enumerate(cols)}

    src_idx, dst_idx, weights = [], [], []

    for src in cols:
        series = mi_dict[src]

        # 우리 그래프에 존재하는 변수만 남기고, 자기 자신은 제외
        series = series[series.index.isin(cols)]
        if src in series.index:
            series = series.drop(index=src)

        # 상위 k개 선택 (이미 내림차순 정렬되어 있다고 가정)
        top_neighbors = series.head(top_k)

        for dst, w in top_neighbors.items():
            src_idx.append(col_to_idx[src])
            dst_idx.append(col_to_idx[dst])
            if return_edge_attr:
                weights.append(float(w))

    edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)

    if return_edge_attr:
        edge_attr = torch.tensor(weights, dtype=torch.float)
        return edge_index, edge_attr

    return edge_index


In [27]:
edge_index = build_edge_index_from_mi_directed(mi_dict, top_k=7, return_edge_attr=False)

In [28]:
edge_index.shape

torch.Size([2, 511])